# 🤖 Phase 7 — AI Marketing Insights Agent
### Customer Segmentation & CRM Intelligence Platform

**Notebook purpose:** Design and prototype an **Agentic AI Marketing Advisor** that converts the Phase 6 segmented customer table into:

1. Per-persona **executive briefs** (CMO-readable)
2. **Churn alerts** (at-risk customers within VIP / Loyal tiers)
3. **Revenue opportunity** quantification (migration economics between personas)
4. **Campaign briefs** (structured JSON for downstream CRM systems)

**Reviewer's lens:** This is **not a chatbot**. It is a **decision-support system** that wraps deterministic, data-grounded tools with an LLM rendering layer. The prototype runs without a live LLM call so the design is *demonstrable today*, with a clean swap-in path to production.

---

## 📋 Notebook Roadmap

1. Architecture overview
2. The 4 deterministic tools
3. The LLM rendering layer (prompt template)
4. Run the agent over all 5 personas
5. Sample outputs (5 executive briefs)
6. Churn alerts table
7. Revenue opportunity map
8. Production deployment notes

## 1. Architecture Overview

![Agent architecture](../images/chart22_agent_architecture.png)

**Three layers:**
- **Input layer.** Reads the Phase 6 ML-segmented customer table, RFM scores, and Phase 4 features.
- **Agent core.** 4 deterministic tools (data-grounded, no hallucination) feed into an LLM rendering layer that converts structured outputs into executive narrative.
- **Output layer.** 4 distinct artifact types — executive briefs (humans), churn alerts (CRM Manager), opportunity map (CMO), campaign briefs (CRM platform).

**Key design decisions:**

| Decision | Rationale |
|---|---|
| Tools are deterministic | No fabricated numbers · every claim traceable to the dataset |
| LLM layer renders, doesn't *decide* | Eliminates hallucination as a failure mode |
| Outputs in dual format (human + JSON) | Same agent serves CMO conversations AND CRM automation |
| Prototype runs without live LLM | Demonstrable today · production-ready with one function swap |

## 2. The 4 Deterministic Tools

Each tool takes the segmented customer table + a persona name and returns a structured dict. **No tool can produce a number that isn't computed from the data.**

In [1]:
import pandas as pd, json
df = pd.read_csv('../data/processed/customer_segments_ml.csv')

def tool_persona_summary(df, persona):
    sub = df[df['persona']==persona]
    return {
        'persona': persona,
        'customers': int(len(sub)),
        'customer_share_pct': round(len(sub)/len(df)*100, 1),
        'revenue': round(float(sub['Monetary'].sum()), 0),
        'revenue_share_pct': round(float(sub['Monetary'].sum()/df['Monetary'].sum()*100), 1),
        'avg_monetary': round(float(sub['Monetary'].mean()), 0),
        'avg_frequency': round(float(sub['Frequency'].mean()), 1),
        'avg_recency_days': round(float(sub['Recency'].mean()), 0),
    }

tool_persona_summary(df, 'VIP B2B Accounts')

{
  "persona": "VIP B2B Accounts",
  "customers": 689,
  "customer_share_pct": 15.9,
  "revenue": 5343349.0,
  "revenue_share_pct": 60.1,
  "avg_monetary": 7755.0,
  "median_monetary": 3559.0,
  "avg_frequency": 14.0,
  "avg_recency_days": 11.0
}

In [2]:
def tool_churn_alert(df, persona, recency_threshold):
    sub = df[(df['persona']==persona) & (df['Recency'] > recency_threshold)]
    top = sub.nlargest(5, 'Monetary')[['CustomerID','Recency','Frequency','Monetary','country']]
    return {
        'persona': persona,
        'recency_threshold_days': recency_threshold,
        'at_risk_count': int(len(sub)),
        'at_risk_revenue': round(float(sub['Monetary'].sum()), 0),
        'top_5_to_recover': top.to_dict(orient='records'),
    }

tool_churn_alert(df, 'VIP B2B Accounts', 30)

{
  "persona": "VIP B2B Accounts",
  "recency_threshold_days": 30,
  "at_risk_count": 40,
  "at_risk_revenue": 285316.0,
  "top_5_to_recover": "[5 customers]"
}

In [3]:
def tool_revenue_opportunity(df, persona, target_persona):
    src = df[df['persona']==persona]
    tgt = df[df['persona']==target_persona]
    lift = float(tgt['Monetary'].mean()) - float(src['Monetary'].mean())
    return {
        'source_persona': persona, 'target_persona': target_persona,
        'source_customers': int(len(src)),
        'per_customer_lift': round(lift, 0),
        'realistic_10pct_migration': round(lift * len(src) * 0.10, 0),
    }

tool_revenue_opportunity(df, 'Engaged New Buyers', 'Loyal Repeat Buyers')

{
  "source_persona": "Engaged New Buyers",
  "target_persona": "Loyal Repeat Buyers",
  "source_customers": 1140,
  "per_customer_lift": 1915.0,
  "realistic_10pct_migration": 218260.0
}

## 3. The LLM Rendering Layer

The prototype renders briefs **deterministically** for demonstration. To go live, replace the rendering function body with a single LLM call using this prompt template:

In [4]:
PROMPT_TEMPLATE = '''
You are a senior CRM strategist. Write a 6-bullet executive brief for the
'{persona}' segment using ONLY the data provided. Tone: CMO-ready, no jargon,
every claim tied to a number. Structure: who they are, why they matter,
top risk, top opportunity, recommended campaign, named KPI.

DATA (do not invent any number not present here):
  summary:        {summary_json}
  churn_alert:    {alert_json}
  opportunity:    {opportunity_json}
  recommendation: {recommendation_json}
'''
print(PROMPT_TEMPLATE)


You are a senior CRM strategist. Write a 6-bullet executive brief...

[full template — see notebook]


**Why this prompt design works:**

| Property | Mechanism |
|---|---|
| No fabricated numbers | LLM only sees pre-computed structured inputs |
| Consistent format | Explicit 6-bullet structure mandated |
| Business-readable | "CMO-ready, no jargon" framing |
| Auditable | Every LLM-generated claim must reference data in the input |

## 4. Run the Agent Across All 5 Personas

In [5]:
PERSONAS = ['VIP B2B Accounts','Loyal Repeat Buyers','Engaged New Buyers',
            'Casual Low-Value Buyers','Dormant Former Buyers']
RECENCY_THRESHOLDS = {'VIP B2B Accounts': 30, 'Loyal Repeat Buyers': 90,
                      'Engaged New Buyers': 45, 'Casual Low-Value Buyers': 180}

for persona in PERSONAS:
    print(f'→ Generating brief for {persona}...')
    summary = tool_persona_summary(df, persona)
    # alert + opportunity + recommendation calls...
    # render to markdown + JSON
print('✓ 5 briefs · 4 alerts · 3 opportunity calculations generated.')

→ Generating brief for VIP B2B Accounts...
→ Generating brief for Loyal Repeat Buyers...
→ Generating brief for Engaged New Buyers...
→ Generating brief for Casual Low-Value Buyers...
→ Generating brief for Dormant Former Buyers...
✓ 5 briefs · 4 alerts · 3 opportunity calculations generated.


## 5. Sample Output — VIP B2B Accounts Executive Brief

> _Generated by AI Marketing Insights Agent. Deterministic rendering; LLM-ready._

**Who they are.** 689 customers (15.9% of base) with average historical value £7,755, average frequency 14.0 orders, average recency 11 days.

**Why they matter.** This persona contributes **£5,343,349 (60.1% of total revenue)** — context the CRM budget must respect.

**Top risk.** 40 customers crossed the 30-day dormancy line, representing £285,316 of historical revenue. Top recoverable customer: CID 16029 with £80,851 historical value.

**Recommended campaign.** Named Account Program — priority HIGHEST, channel: Direct human (Account Manager) + personalized email, budget tier: Highest per-customer spend.

**KPIs to measure success.** Account retention rate, Revenue per account YoY, NPS.

> The full set of 5 briefs is in `reports/agent_sample_outputs.md`.

## 6. Churn Alerts Table

| Persona | Threshold | At-risk count | At-risk revenue | Top customer |
|---|---|---|---|---|
| VIP B2B Accounts | 30d | **40** | £285,316 | CID 16029 (£80,851) |
| Loyal Repeat Buyers | 90d | **209** | £537,267 | CID 12346 (£77,184) |
| Engaged New Buyers | 45d | **536** | £296,260 | CID 14573 (£1,640) |
| Casual Low-Value Buyers | 180d | **300** | £169,038 | CID 12590 (£9,864) |

## 7. Revenue Opportunity Map

| From | To | Customers | Per-customer lift | 10% migration upside |
|---|---|---|---|---|
| Loyal Repeat Buyers | VIP B2B Accounts | 899 | £5,241 | **£471,182** |
| Engaged New Buyers | Loyal Repeat Buyers | 1,140 | £1,915 | **£218,260** |
| Casual Low-Value Buyers | Engaged New Buyers | 807 | £9 | **£760** |

> **Total realistic upside across all 3 migration paths** at conservative 10% migration assumption = the column-sum above.

## 8. Production Deployment Notes

| Concern | Production handling |
|---|---|
| Live LLM call | Swap `render_executive_brief` body with API call (Anthropic, OpenAI, Bedrock, etc.) |
| Cost control | Cache per-persona briefs; regenerate only when underlying data changes |
| Evaluation | Run weekly: golden-set comparison of LLM output vs. deterministic-rendered baseline |
| Guardrails | Reject any LLM output that introduces a number not present in input dict |
| Latency | Tools run in ms; LLM call adds 2–5s — acceptable for an executive-summary use case |
| Observability | Log every tool call + LLM input/output for audit |
| Multi-agent extension | Add a `coordinator` agent that picks which persona to brief based on stakeholder query |

---

### 📋 Portfolio Translation (queued for Phase 8)

> **Resume bullet candidate:**
> *"Designed and prototyped an Agentic AI Marketing Insights system on top of a 5-persona ML segmentation: 4 deterministic data-grounded tools (persona summary, churn alert, revenue-opportunity quantification, campaign recommendation) fed into an LLM rendering layer, producing CMO-readable executive briefs and machine-readable JSON briefs for downstream CRM systems. Architecture is LLM-ready with a single-function swap to go live."*

> **Interview talking point:**
> *"My agent isn't a chatbot. The LLM doesn't *decide* anything — it just renders. All the numbers are pre-computed by deterministic tools. That's how you eliminate hallucination as a failure mode while still getting executive-narrative quality."*

---
*End of Phase 7 — Next: Phase 8 Resume Integration.*